# Password Classification Using Machine Learning
Classify passwords as **Weak (0)**, **Medium (1)** or **Strong (2)** using features: length, uppercase, lowercase, digits and special characters.

## Q1. Dataset Collection
Download the *Password Strength Classifier Dataset* (Bhavik Bansal) from Kaggle.

In [ ]:
!pip -q install kagglehub
import kagglehub, os
path = kagglehub.dataset_download("bhavikbb/password-strength-classifier-dataset")
print("Downloaded to:", path)
print(os.listdir(path))

Using Colab cache for faster access to the 'password-strength-classifier-dataset' dataset.
Downloaded to: /kaggle/input/password-strength-classifier-dataset
['data.csv']


## Q2/Q3. Read the dataset and delete the first (Index/ID) column

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv(os.path.join(path, "data.csv"), on_bad_lines="skip", engine="python")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (669640, 2)
Columns: ['password', 'strength']


,password,strength
0,kzde5577,1
1,kino3434,1
2,visi7k1yr,1
3,megzy123,1
4,lamborghin1,1


In [ ]:
# Delete the first column if it is an Index/ID column (not 'password' or 'strength')
first = df.columns[0]
if first.lower() not in ("password", "strength"):
    df = df.drop(columns=[first])
    print("Dropped index column:", first)
else:
    print("First column is", repr(first), "- this file has no separate Index/ID column, nothing to drop.")
print("Columns now:", list(df.columns))

First column is 'password' - this file has no separate Index/ID column, nothing to drop.
Columns now: ['password', 'strength']


## Q3. Data Cleaning
Remove null values, duplicates and irrelevant data.

In [ ]:
before = len(df)
print("Null values per column:\n", df.isnull().sum())
df = df.dropna()
print("After removing nulls:", len(df))

df["password"] = df["password"].astype(str)
df = df.drop_duplicates(subset="password")
print("After removing duplicates:", len(df))

# Irrelevant data: invalid labels, empty or whitespace-only passwords
df["strength"] = pd.to_numeric(df["strength"], errors="coerce")
df = df[df["strength"].isin([0, 1, 2])]
df = df[df["password"].str.strip().str.len() > 0]
df["strength"] = df["strength"].astype(int)
print("After removing irrelevant rows:", len(df), f"(removed {before-len(df)} in total)")
print(df["strength"].value_counts().sort_index())
df.to_csv("cleaned_data.csv", index=False)

Null values per column:
 password    1
strength    0
dtype: int64
After removing nulls: 669639
After removing duplicates: 669639
After removing irrelevant rows: 669639 (removed 1 in total)
strength
0     89701
1    496801
2     83137
Name: count, dtype: int64


## Q4. Feature Extraction

In [ ]:
import string
def extract_features(pw):
    pw = str(pw)
    return [
        len(pw),                                   # length
        sum(c.isupper() for c in pw),              # uppercase
        sum(c.islower() for c in pw),              # lowercase
        sum(c.isdigit() for c in pw),              # numeric
        sum(not c.isalnum() for c in pw),          # special characters
    ]
FEATURES = ["length", "uppercase", "lowercase", "numeric", "special"]
X = pd.DataFrame(df["password"].map(extract_features).tolist(), columns=FEATURES)
y = df["strength"].values
X.head()

,length,uppercase,lowercase,numeric,special
0,8,0,4,4,0
1,8,0,4,4,0
2,9,0,7,2,0
3,8,0,5,3,0
4,11,0,10,1,0


## Q4. Model Training
Stratified train/test split (80/20). Duplicates were removed **before** splitting, so the same password cannot be in both sets (no data leakage).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)
model = RandomForestClassifier(n_estimators=50, max_depth=15, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Training done")

Train: (535711, 5) Test: (133928, 5)
Training done


## Q5. Model Testing

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=["Weak","Medium","Strong"]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Test accuracy: 100.00%
              precision    recall  f1-score   support

        Weak       1.00      1.00      1.00     17940
      Medium       1.00      1.00      1.00     99360
      Strong       1.00      1.00      1.00     16628

    accuracy                           1.00    133928
   macro avg       1.00      1.00      1.00    133928
weighted avg       1.00      1.00      1.00    133928

Confusion matrix:
 [[17940     0     0]
 [    0 99360     0]
 [    0     0 16628]]


## Q6. Model Saving (.pkl)

In [ ]:
import pickle
with open("password_model.pkl", "wb") as f:
    pickle.dump(model, f)
print("Saved password_model.pkl,", round(os.path.getsize("password_model.pkl")/1e6, 2), "MB")

# Verify: load the saved model and predict
with open("password_model.pkl", "rb") as f:
    loaded = pickle.load(f)
labels = {0: "Weak", 1: "Medium", 2: "Strong"}
for pw in ["abc", "kzde5577", "Hello@2024", "Xy7#pQ9!mZ2&vL"]:
    p = loaded.predict(pd.DataFrame([extract_features(pw)], columns=FEATURES))[0]
    print(f"{pw!r:20} -> {labels[p]}")

Saved password_model.pkl, 0.21 MB
'abc'                -> Weak
'kzde5577'           -> Medium
'Hello@2024'         -> Medium
'Xy7#pQ9!mZ2&vL'     -> Strong


In [ ]:
# Download the model and cleaned dataset to your computer
from google.colab import files
files.download("password_model.pkl")
files.download("cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>